# Bibliotecas
- Google API e acesso HTTP



In [10]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from abc import ABC, abstractmethod

# Padrão de Projeto
- Foi usado o Singleton para a conexão com a API

    

In [13]:
# ==========================================
# PADRÃO 1: SINGLETON (Conexão com a API)
# ==========================================
class GoogleSheetsConnection:
    _instance = None
    _sheet = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super(GoogleSheetsConnection, cls).__new__(cls)
            # Escopo necessário para acessar Drive e Sheets
            scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
            
            # Autenticação (certifique-se de que o credentials.json está na pasta)
            creds = ServiceAccountCredentials.from_json_keyfile_name("credentials.json", scope)
            client = gspread.authorize(creds)
            
            # ID da planilha da roboAp 
            SPREADSHEET_ID = "1ln-zOX2YsysEhJhVF1yTPwuEnXGfHkB23KH6Ao10ODA/edit?pli=1&gid=0#gid=0"
            cls._sheet = client.open_by_key(SPREADSHEET_ID).sheet1
            print("[SISTEMA] Conexão segura estabelecida com o Google Sheets da equipe.")
        return cls._instance

def ler_planilha(self):
        # Retorna todos os dados ignorando a linha de cabeçalho
        return self._sheet.get_all_records()

def atualizar_quantidade(self, item_id, coluna_qtd, nova_qtd):
    # Procura a célula que contém o ID da peça
    celula = self._sheet.find(item_id)
    if celula:
        # Atualiza apenas a célula específica da quantidade na linha encontrada
        self._sheet.update_cell(celula.row, coluna_qtd, nova_qtd)
        return True
    return False

# Padrão de Arquitetura
- Foi usado o padrão MVC (View, ModeL, Controller)


In [11]:
# ==========================================
# CAMADA MODEL
# ==========================================
class InventarioModel:
    def __init__(self):
        self.db = GoogleSheetsConnection()
        self.COLUNA_QTD = 5 # A coluna 'QTD' é a 5ª coluna na planilha (A=1, B=2...)

    def obter_estoque(self):
        # Puxa os dados brutos da API
        registros = self.db.ler_planilha()
        estoque = {}
        
        # Formata os dados para o padrão que o sistema espera
        for linha in registros:
            # Lida com possíveis linhas vazias
            if not linha.get('ID'): 
                continue
                
            estoque[str(linha['ID']).strip()] = {
                "nome": linha.get('NOME', 'N/A'),
                "categoria": linha.get('CATEGORIA', 'N/A'),
                "responsavel": linha.get('RESPONSAVEL', 'N/A'),
                "qtd": int(linha.get('QTD', 0)),
                "status": linha.get('STATUS', 'N/A')
            }
        return estoque

    def registrar_falha_componente(self, item_id, descricao):
        estoque_atual = self.obter_estoque()
        
        # Validação da regra de negócio: a peça existe e tem mais de 0 unidades?
        if item_id in estoque_atual and estoque_atual[item_id]["qtd"] > 0:
            nova_quantidade = estoque_atual[item_id]["qtd"] - 1
            # Envia a ordem de gravação para a API do Google
            return self.db.atualizar_quantidade(item_id, self.COLUNA_QTD, nova_quantidade)
            
        return False

# ==========================================
# CAMADA VIEW
# ==========================================
class TerminalView:
    def exibir_estoque(self, dados):
        print("\n=== INVENTÁRIO (Sincronizado via Google Sheets) ===")
        print(f"{'ID':<12} | {'NOME':<24} | {'CATEGORIA':<12} | {'RESPONSÁVEL':<15} | {'QTD':<4} | {'STATUS'}")
        print("-" * 88)
        
        if not dados:
            print("Nenhum dado encontrado. Verifique se a planilha tem informações.")
        else:
            for cod, info in dados.items():
                print(f"{cod:<12} | {info['nome']:<24} | {info['categoria']:<12} | {info['responsavel']:<15} | {info['qtd']:<4} | {info['status']}")
        print("========================================================================================\n")

    def coletar_input(self, mensagem):
        return input(mensagem)

    def exibir_mensagem(self, mensagem):
        print(f"\n>>> {mensagem}\n")

# ==========================================
# PADRÃO 2: COMMAND (Comportamental)
# ==========================================
class Command(ABC):
    @abstractmethod
    def execute(self):
        pass

class VisualizarEstoqueCommand(Command):
    def __init__(self, model, view):
        self.model = model
        self.view = view

    def execute(self):
        self.view.exibir_mensagem("Buscando dados na nuvem, aguarde...")
        dados = self.model.obter_estoque()
        self.view.exibir_estoque(dados)

class RegistrarFalhaCommand(Command):
    def __init__(self, model, view):
        self.model = model
        self.view = view

    def execute(self):
        item_id = self.view.coletar_input("Digite o ID do componente que falhou/queimou: ").upper().strip()
        desc = self.view.coletar_input("Descreva o motivo da queima ou problema técnico: ")
        
        self.view.exibir_mensagem("Enviando requisição para a nuvem, aguarde...")
        sucesso = self.model.registrar_falha_componente(item_id, desc)
        
        if sucesso:
            self.view.exibir_mensagem(f"SUCESSO: Falha registrada no sistema! 1 unidade de '{item_id}' foi deduzida da planilha.")
        else:
            self.view.exibir_mensagem("ERRO: O ID informado não existe na planilha ou a quantidade em estoque já é zero.")

# ==========================================
# CAMADA CONTROLLER
# ==========================================
class EstoqueController:
    def __init__(self, model, view):
        self.comandos = {
            "1": VisualizarEstoqueCommand(model, view),
            "2": RegistrarFalhaCommand(model, view)
        }
        self.view = view

    def iniciar(self):
        while True:
            print("=== SISTEMA DE GESTÃO DE HARDWARE ===")
            print("1. Visualizar Estoque Geral")
            print("2. Registrar Falha/Queima de Componente")
            print("0. Sair do Sistema")
            opcao = self.view.coletar_input("Escolha uma opção numérica: ")
            
            if opcao == "0":
                print("\nEncerrando a conexão e saindo do sistema...")
                break
            elif opcao in self.comandos:
                self.comandos[opcao].execute()
            else:
                self.view.exibir_mensagem("Opção inválida. Tente novamente.")


In [14]:
# ==========================================
# PONTO DE ENTRADA DO PROGRAMA
# ==========================================
if __name__ == "__main__":
    # Inicializa as camadas e injeta as dependências
    app_model = InventarioModel()
    app_view = TerminalView()
    app_controller = EstoqueController(app_model, app_view)
    
    # Roda a aplicação
    app_controller.iniciar()

SpreadsheetNotFound: <Response [404]>